# Principal Angle Analysis: SALTEDORA-V4 vs DoRA

This notebook analyses how SALTEDORA-V4 at different **energy thresholds** aligns with DoRA (treated as the full fine-tuning reference) by computing **principal angles between learned weight-update subspaces (ΔW)**.

### What are principal angles?
Given two matrices A and B, we extract their column-space bases (via SVD) and compute the angles between those subspaces. A mean angle near **0° means highly similar** adaptation directions; near **90° means orthogonal** (completely different).

### Structure
| Section | What it covers |
|---|---|
| 1 | Torch-free `.pt` file loader + weight reconstruction |
| 2 | Principal angle utilities |
| 3 | Path config |
| 4 | Analysis A – Final weights: all thresholds vs DoRA |
| 5 | Analysis B – Epoch dynamics (single threshold) |
| 6 | Analysis C – All thresholds, per epoch |
| 7 | Plots |
| 8 | Insights summary |

---
## 1 · Torch-free `.pt` loader and weight reconstruction

In [ ]:
import os, sys, zipfile, pickle, types, gc
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.gridspec import GridSpec
import warnings
warnings.filterwarnings('ignore')

# ── Mock torch so pickle can deserialise .pt files without a real torch install ──
_torch      = types.ModuleType('torch')
_torch_utils = types.ModuleType('torch._utils')
sys.modules['torch']        = _torch
sys.modules['torch._utils'] = _torch_utils

_DTYPES = {
    'FloatStorage':    np.float32,
    'DoubleStorage':   np.float64,
    'HalfStorage':     np.float16,
    'BFloat16Storage': np.float32,
    'LongStorage':     np.int64,
    'IntStorage':      np.int32,
    'ShortStorage':    np.int16,
    'ByteStorage':     np.uint8,
    'BoolStorage':     np.bool_,
}
for _k in _DTYPES:
    setattr(_torch, _k, type(_k, (), {'__name__': _k}))

def _dtype(name):
    s = str(name)
    for k, v in _DTYPES.items():
        if k in s: return v
    return np.float32


class _Stub:
    """Lazy numpy-backed tensor stub (avoids materialising until needed)."""
    __slots__ = ('key','offset','shape','stride','dtype','_store')
    def __init__(self, key, offset, shape, stride, dtype, store):
        self.key=key; self.offset=offset; self.shape=shape
        self.stride=stride; self.dtype=dtype; self._store=store
    def np(self):
        arr = self._store[self.key]
        if not self.shape: return arr[self.offset]
        n = 1
        for s in self.shape: n *= s
        return arr[self.offset : self.offset + n].reshape(self.shape)


def load_pt(path):
    """Load a PyTorch checkpoint as nested Python dicts / _Stub tensors."""
    z      = zipfile.ZipFile(path)
    prefix = z.namelist()[0].split('/')[0]
    store  = {}

    class _U(pickle.Unpickler):
        def find_class(self, mod, name):
            if mod == 'torch._utils' and name == '_rebuild_tensor_v2':
                def _r(storage, off, size, stride, rg, hooks):
                    return _Stub(storage[0], off, tuple(size),
                                 tuple(stride), storage[1], store)
                return _r
            if mod == 'torch._utils' and name == '_rebuild_from_type_v2':
                return lambda f, t, a, s: f(*a)
            if mod == 'torch'       and name == 'Size':        return tuple
            if mod == 'collections' and name == 'OrderedDict': return dict
            try:   return super().find_class(mod, name)
            except: return lambda *a, **k: None

        def persistent_load(self, pid):
            _, dc, key, loc, numel = pid
            dt = _dtype(dc.__name__ if hasattr(dc,'__name__') else str(dc))
            store[key] = np.frombuffer(
                z.read(f'{prefix}/data/{key}'), dtype=dt).copy()
            return (key, dt)

    with z.open(f'{prefix}/data.pkl') as f:
        return _U(f).load()

In [ ]:
# ── SALTEDORA-V4 forward pass (from models.py) ────────────────────────────────
#
#  W_eff = U_top  diag( relu(S_top * α + β) )  Vh_top        # head (learned)
#        + U_tail diag( S_tail )                Vh_tail        # tail base (frozen)
#        + (U_tail_r * S_tail_r) * relu(D)  @  R @ Vh_tail_r  # intrinsic delta
#
#  W_base = U_top diag(S_top) Vh_top + U_tail diag(S_tail) Vh_tail
#
#  ΔW = W_eff - W_base
#     = U_top diag( relu(S_top*α+β) - S_top ) Vh_top          # head delta
#     + (U_tail_r * S_tail_r) * relu(D) @ R @ Vh_tail_r        # tail delta

def salt_delta_W(layer_dict):
    """Reconstruct ΔW for one SALTEDoRA-V4 layer."""
    sd = layer_dict['state_dict_no_weight']
    t  = lambda k: sd[k].np().astype(np.float32)

    alpha = t('alpha');  beta   = t('beta');   S_top  = t('S_top')
    U_top = t('U_top');  Vh_top = t('Vh_top')
    D     = t('D');      R      = t('R')
    U_r   = t('U_tail_r'); S_r  = t('S_tail_r'); Vh_r = t('Vh_tail_r')

    # Head delta
    dsigma  = np.maximum(0., S_top * alpha + beta) - S_top
    dW_head = (U_top * dsigma) @ Vh_top

    # Intrinsic tail delta
    B       = (U_r * S_r) * np.maximum(0., D)
    dW_tail = B @ R @ Vh_r

    return dW_head + dW_tail


# ── DoRA ΔW ──────────────────────────────────────────────────────────────────
#  DoRA stores the fully merged effective weight in 'weight'.
#  ΔW = effective_weight - base_layer_weight

def dora_delta_W(layer_dict):
    """Reconstruct ΔW for one DoRA layer."""
    W_eff  = layer_dict['weight'].np().astype(np.float32)
    W_base = layer_dict['state_dict_no_weight']['base_layer.weight'].np().astype(np.float32)
    return W_eff - W_base


def r_top_of(layer_dict):
    """Number of head singular values (energy-threshold-determined)."""
    return layer_dict['state_dict_no_weight']['alpha'].np().shape[0]


# ── Convenience loaders (return per-layer dicts, then free memory) ───────────

def load_dora_dWs(path):
    data = load_pt(path)['qkv']
    out  = {l: dora_delta_W(v) for l, v in data.items()}
    del data; gc.collect()
    return out

def load_salt_dWs(path):
    """Returns (dW_dict, rtop_dict) or (None, None) if file missing."""
    if not os.path.isfile(path): return None, None
    data  = load_pt(path)['qkv']
    dws   = {l: salt_delta_W(v) for l, v in data.items()}
    rtops = {l: r_top_of(v)     for l, v in data.items()}
    del data; gc.collect()
    return dws, rtops

print('Loaders ready.')

---
## 2 · Principal angle utilities

In [ ]:
def col_basis(M, k):
    """
    Orthonormal basis for the column space of M.
    Returns top-k left singular vectors (shape: M.shape[0] x k).
    """
    U, _, _ = np.linalg.svd(M.astype(np.float64), full_matrices=False)
    return U[:, :k]


def principal_angles_deg(A, B, k):
    """
    Mean principal angle (°) between the column spaces of A and B.

    Algorithm:
        1. Compute orthonormal bases Q_A, Q_B  (via SVD)
        2. SVD of Q_A^T Q_B  →  singular values σ_i = cos(θ_i)
        3. θ_i = arccos(σ_i)
    """
    Qa = col_basis(A, k)
    Qb = col_basis(B, k)
    s  = np.linalg.svd(Qa.T @ Qb, compute_uv=False)
    s  = np.clip(s[:k], 0., 1.)
    return float(np.mean(np.degrees(np.arccos(s))))


def all_principal_angles_deg(A, B, k):
    """Return all k principal angles (not just the mean)."""
    Qa = col_basis(A, k)
    Qb = col_basis(B, k)
    s  = np.linalg.svd(Qa.T @ Qb, compute_uv=False)
    return np.degrees(np.arccos(np.clip(s[:k], 0., 1.)))


def cos2_alignment(A, B, k):
    """
    Mean cos²(θ) between the k-dim subspaces.  1 = identical, 0 = orthogonal.
    """
    Qa = col_basis(A, k)
    Qb = col_basis(B, k)
    s  = np.linalg.svd(Qa.T @ Qb, compute_uv=False)
    return float(np.mean(np.clip(s[:k], 0., 1.) ** 2))


def grassmann_distance(A, B, k):
    """
    Grassmann distance = sqrt( sum of squared principal angles )
    between the k-dim subspaces.
    """
    angles_rad = np.radians(all_principal_angles_deg(A, B, k))
    return float(np.sqrt(np.sum(angles_rad**2)))


def effective_rank(M, thr=0.99):
    """
    Minimum number of singular values whose cumulative energy >= thr.
    """
    sv  = np.linalg.svd(M.astype(np.float64), compute_uv=False)
    e   = sv**2
    cum = np.cumsum(e) / e.sum()
    return int(np.searchsorted(cum, thr)) + 1


def layer_short(name):
    """Convert full layer name to short label, e.g. 'L3-Q'."""
    parts = name.split('.')
    try:
        li  = parts.index('layer')
        idx = parts[li + 1]
        k   = parts[-1][0].upper()   # query/key/value → Q/K/V
        return f'L{idx}-{k}'
    except:
        return name[-12:]

print('Geometry utilities ready.')

---
## 3 · Path configuration

**Edit `BASE` to match your local folder structure.**

In [ ]:
# ── Edit these to match your paths ───────────────────────────────────────────
BASE      = r'C:\Users\Tay Han\OneDrive\Documents\bert-base-uncased'   # <-- change me
OUT_DIR   = r'C:\Users\Tay Han\OneDrive\Documents\principal_angle_results'  # where to save figures

os.makedirs(OUT_DIR, exist_ok=True)

DORA_DIR  = os.path.join(BASE, 'dora', 'r_8', 'et_na', 'glue_sst2', 'weights')
SALT_BASE = os.path.join(BASE, 'saltedora_v4')

# All available thresholds and ranks
THRESHOLDS = [0.40, 0.50, 0.60, 0.70, 0.80, 0.90]
EPOCHS     = [1, 2, 3, 4, 5]

# Subspace dimensionality for principal angle computation
# (k = number of singular vectors to compare)
K = 8

# Primary rank to analyse
R = 128


def dora_pt(epoch_or_final='final'):
    fname = 'final.pt' if epoch_or_final == 'final' else f'epoch_{epoch_or_final}.pt'
    return os.path.join(DORA_DIR, fname)


def salt_pt(r, et, epoch_or_final='final'):
    fname = 'final.pt' if epoch_or_final == 'final' else f'epoch_{epoch_or_final}.pt'
    return os.path.join(SALT_BASE, f'r_{r}', f'et_{et:.2f}',
                        'glue_sst2', 'weights', fname)


# Quick sanity check
print('DoRA final path exists:', os.path.isfile(dora_pt()))
print('SALT r=128 et=0.70 final exists:',
      os.path.isfile(salt_pt(128, 0.70, 'final')))

---
## 4 · Analysis A — Final weights: all thresholds vs DoRA

For each energy threshold, load the **final** SALTEDORA checkpoint and compute per-layer:
- Mean principal angle vs DoRA ΔW  
- Cos² alignment  
- r_top (# head singular values used)  
- Effective rank of ΔW

In [ ]:
# Load DoRA final once as reference
print('Loading DoRA final …')
dora_dWs = load_dora_dWs(dora_pt('final'))
LAYERS   = list(dora_dWs.keys())
N        = len(LAYERS)
T        = len(THRESHOLDS)
print(f'  {N} layers found:', [layer_short(l) for l in LAYERS])

# Output arrays  (threshold × layer)
angle_mat = np.full((T, N), np.nan)
align_mat = np.full((T, N), np.nan)
rtop_mat  = np.full((T, N), np.nan)
erank_mat = np.full((T, N), np.nan)

print(f'\nComputing principal angles  [k={K}] …')
for ti, et in enumerate(THRESHOLDS):
    salt_dws, rtops = load_salt_dWs(salt_pt(R, et, 'final'))
    if salt_dws is None:
        print(f'  [SKIP] r={R} et={et:.2f} – file not found'); continue

    for li, layer in enumerate(LAYERS):
        dw_d = dora_dWs[layer]
        dw_s = salt_dws.get(layer)
        if dw_s is None: continue
        try:
            angle_mat[ti, li] = principal_angles_deg(dw_d, dw_s, K)
            align_mat[ti, li] = cos2_alignment(dw_d, dw_s, K)
            rtop_mat [ti, li] = rtops.get(layer, 0)
            erank_mat[ti, li] = effective_rank(dw_s)
        except Exception as e:
            print(f'    Error at {layer_short(layer)}: {e}')

    del salt_dws, rtops; gc.collect()
    print(f'  et={et:.2f}  mean_angle={np.nanmean(angle_mat[ti]):.2f}°  '
          f'alignment={np.nanmean(align_mat[ti]):.4f}  '
          f'mean_rtop={np.nanmean(rtop_mat[ti]):.1f}')

print('\nDone.')

---
## 5 · Analysis B — Epoch dynamics (single threshold)

Track how the SALTEDORA subspace **moves toward / away from DoRA** across training epochs.

In [ ]:
ET_DYN = 0.70   # threshold to inspect epoch-by-epoch

to_dora_mat = np.full((len(EPOCHS), N), np.nan)   # epoch × layer (angle vs DoRA FINAL)
consec_mat  = np.full((len(EPOCHS)-1, N), np.nan) # epoch × layer (angle between consecutive epochs)

prev_dws = None
for ei, epoch in enumerate(EPOCHS):
    salt_dws, _ = load_salt_dWs(salt_pt(R, ET_DYN, epoch))
    if salt_dws is None:
        print(f'  [SKIP] epoch={epoch}'); prev_dws = None; continue

    for li, layer in enumerate(LAYERS):
        dw_d = dora_dWs[layer]
        dw_s = salt_dws.get(layer)
        if dw_s is None: continue
        try:
            to_dora_mat[ei, li] = principal_angles_deg(dw_d, dw_s, K)
        except: pass
        if prev_dws and layer in prev_dws:
            try:
                consec_mat[ei-1, li] = principal_angles_deg(prev_dws[layer], dw_s, K)
            except: pass

    prev_dws = salt_dws
    print(f'  Epoch {epoch}  →  mean angle to DoRA final = {np.nanmean(to_dora_mat[ei]):.2f}°')

print('Done.')

---
## 6 · Analysis C — All thresholds × all epochs

Creates a **threshold × epoch** matrix of mean principal angles — shows how the training trajectory differs across energy thresholds.

In [ ]:
et_epoch_mat = np.full((T, len(EPOCHS)), np.nan)

print(f'Computing  [k={K}] …')
for ti, et in enumerate(THRESHOLDS):
    for ei, epoch in enumerate(EPOCHS):
        salt_dws, _ = load_salt_dWs(salt_pt(R, et, epoch))
        if salt_dws is None: continue
        angs = []
        for layer in LAYERS:
            dw_d = dora_dWs[layer]
            dw_s = salt_dws.get(layer)
            if dw_s is not None:
                try: angs.append(principal_angles_deg(dw_d, dw_s, K))
                except: pass
        del salt_dws; gc.collect()
        et_epoch_mat[ti, ei] = np.nanmean(angs) if angs else np.nan
    print(f'  et={et:.2f}: epochs = {np.round(et_epoch_mat[ti], 2)}')

print('Done.')

---
## 7 · Plots

### 7.1 · Summary 4-panel figure

In [ ]:
COLORS = plt.cm.plasma(np.linspace(0.1, 0.9, T))
xlabs  = [layer_short(l) for l in LAYERS]

fig = plt.figure(figsize=(18, 11))
gs  = GridSpec(2, 2, figure=fig, hspace=0.45, wspace=0.38)

# ── A: mean angle vs ET (final weights) ──────────────────────────────────────
ax = fig.add_subplot(gs[0, 0])
m  = np.nanmean(angle_mat, axis=1)
ax.plot(THRESHOLDS, m, 'o-', color='crimson', lw=2.2, ms=8, label='mean over layers')
ax.fill_between(THRESHOLDS,
                np.nanmin(angle_mat, axis=1),
                np.nanmax(angle_mat, axis=1),
                alpha=0.15, color='crimson', label='layer range')
ax.set_xlabel('Energy Threshold (ET)', fontsize=11)
ax.set_ylabel('Mean Principal Angle (°)', fontsize=11)
ax.set_title('A. ΔW Divergence from DoRA vs ET (final weights)',
             fontsize=11, fontweight='bold')
ax.grid(alpha=0.3); ax.set_xticks(THRESHOLDS); ax.legend(fontsize=9)

# ── B: cos² alignment vs ET ───────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
m2  = np.nanmean(align_mat, axis=1)
ax2.plot(THRESHOLDS, m2, 's-', color='steelblue', lw=2.2, ms=8, label='mean over layers')
ax2.fill_between(THRESHOLDS,
                 np.nanmin(align_mat, axis=1),
                 np.nanmax(align_mat, axis=1),
                 alpha=0.15, color='steelblue', label='layer range')
ax2.set_xlabel('Energy Threshold (ET)', fontsize=11)
ax2.set_ylabel('Cos² Alignment  (1 = identical)', fontsize=11)
ax2.set_title('B. Subspace Alignment with DoRA vs ET',
              fontsize=11, fontweight='bold')
ax2.grid(alpha=0.3); ax2.set_xticks(THRESHOLDS); ax2.legend(fontsize=9)

# ── C: multi-ET epoch dynamics ─────────────────────────────────────────────────
ax3 = fig.add_subplot(gs[1, 0])
for ti, (et, c) in enumerate(zip(THRESHOLDS, COLORS)):
    ax3.plot(EPOCHS, et_epoch_mat[ti], 'o-', color=c,
             lw=1.8, ms=6, label=f'ET={et:.2f}')
ax3.set_xlabel('Epoch', fontsize=11)
ax3.set_ylabel('Mean Angle to DoRA final (°)', fontsize=11)
ax3.set_title('C. Training Dynamics: Subspace Divergence from DoRA',
              fontsize=11, fontweight='bold')
ax3.legend(fontsize=8, loc='best'); ax3.grid(alpha=0.3); ax3.set_xticks(EPOCHS)

# ── D: per-layer heatmap (subsampled every 3rd layer) ──────────────────────────
ax4  = fig.add_subplot(gs[1, 1])
idx  = list(range(0, N, 3))
sub_m = angle_mat[:, idx]
sub_l = [xlabs[i] for i in idx]
im = ax4.imshow(sub_m, aspect='auto', cmap='YlOrRd',
                vmin=0, vmax=90, interpolation='nearest')
ax4.set_yticks(range(T)); ax4.set_yticklabels([f'ET={t:.2f}' for t in THRESHOLDS], fontsize=8)
ax4.set_xticks(range(len(sub_l)))
ax4.set_xticklabels(sub_l, fontsize=7, rotation=60, ha='right')
ax4.set_title('D. Per-layer Principal Angle Heatmap',
              fontsize=11, fontweight='bold')
plt.colorbar(im, ax=ax4, fraction=0.05, label='degrees')

fig.suptitle(f'SALTEDORA-V4 vs DoRA — Principal Angle Analysis\n'
             f'BERT-base-uncased · GLUE-SST2 · r={R} · k={K}',
             fontsize=13, fontweight='bold', y=1.01)

plt.savefig(os.path.join(OUT_DIR, f'fig0_summary_r{R}.png'),
            dpi=150, bbox_inches='tight')
plt.show()

### 7.2 · Full heatmap — angle AND r_top per layer

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(22, 9))

im0 = axes[0].imshow(angle_mat, aspect='auto', cmap='YlOrRd',
                     vmin=0, vmax=90, interpolation='nearest')
axes[0].set_yticks(range(T))
axes[0].set_yticklabels([f'ET={t:.2f}' for t in THRESHOLDS], fontsize=9)
axes[0].set_xticks(range(N))
axes[0].set_xticklabels(xlabs, fontsize=6, rotation=90)
axes[0].set_title(
    f'Mean Principal Angle (°): SALTEDORA-V4 ΔW vs DoRA ΔW  [r={R}, k={K}]\n'
    '0° = identical subspace  |  90° = orthogonal', fontsize=11)
plt.colorbar(im0, ax=axes[0], fraction=0.02, label='degrees')

im1 = axes[1].imshow(rtop_mat, aspect='auto', cmap='Blues', interpolation='nearest')
axes[1].set_yticks(range(T))
axes[1].set_yticklabels([f'ET={t:.2f}' for t in THRESHOLDS], fontsize=9)
axes[1].set_xticks(range(N))
axes[1].set_xticklabels(xlabs, fontsize=6, rotation=90)
axes[1].set_title(
    'r_top per layer and threshold (# head singular values in the frozen top subspace)',
    fontsize=11)
plt.colorbar(im1, ax=axes[1], fraction=0.02, label='r_top')

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, f'fig1_heatmap_r{R}.png'),
            dpi=150, bbox_inches='tight')
plt.show()

### 7.3 · Epoch dynamics — one threshold

In [ ]:
lc = cm.tab20(np.linspace(0, 1, N))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Angle to DoRA final per epoch
for li in range(N):
    ax1.plot(EPOCHS, to_dora_mat[:, li], color=lc[li], alpha=0.28, lw=0.8)
ax1.plot(EPOCHS, np.nanmean(to_dora_mat, axis=1),
         'k-o', lw=2.5, ms=8, label='mean over layers')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Mean Principal Angle (°)')
ax1.set_title(f'Alignment to DoRA (final) per epoch\n[r={R}, ET={ET_DYN}]',
              fontsize=11)
ax1.legend(); ax1.grid(alpha=0.3); ax1.set_xticks(EPOCHS)

# Consecutive-epoch movement
for li in range(N):
    ax2.plot(EPOCHS[1:], consec_mat[:, li], color=lc[li], alpha=0.28, lw=0.8)
ax2.plot(EPOCHS[1:], np.nanmean(consec_mat, axis=1),
         'k-o', lw=2.5, ms=8, label='mean over layers')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Mean Principal Angle (°)')
ax2.set_title(f'Subspace movement epoch-to-epoch\n[r={R}, ET={ET_DYN}]',
              fontsize=11)
ax2.legend(); ax2.grid(alpha=0.3); ax2.set_xticks(EPOCHS[1:])

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, f'fig2_epoch_dynamics_r{R}_et{ET_DYN:.2f}.png'),
            dpi=150, bbox_inches='tight')
plt.show()

### 7.4 · Per-layer bar chart

In [ ]:
idx   = list(range(0, N, 3))
sub_l = [xlabs[i] for i in idx]
sub_m = angle_mat[:, idx]
n_b   = T; x = np.arange(len(idx)); w = 0.8 / n_b

fig, ax = plt.subplots(figsize=(18, 5))
for ti, (et, c) in enumerate(zip(THRESHOLDS, COLORS)):
    off = (ti - n_b/2 + 0.5) * w
    ax.bar(x + off, sub_m[ti], w, label=f'ET={et:.2f}', color=c, alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(sub_l, rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Mean Principal Angle (°)', fontsize=11)
ax.set_title(f'Per-layer Subspace Divergence from DoRA  [r={R}]  (every 3rd layer)',
             fontsize=11)
ax.legend(fontsize=9, ncol=3); ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, f'fig3_per_layer_bar_r{R}.png'),
            dpi=150, bbox_inches='tight')
plt.show()

### 7.5 · r_top vs principal angle scatter

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5.5))
for ti, (et, c) in enumerate(zip(THRESHOLDS, COLORS)):
    v = ~np.isnan(angle_mat[ti])
    ax.scatter(rtop_mat[ti][v], angle_mat[ti][v],
               color=c, alpha=0.55, s=35, label=f'ET={et:.2f}')

ax.set_xlabel('r_top (# head singular values)', fontsize=12)
ax.set_ylabel('Mean Principal Angle to DoRA (°)', fontsize=12)
ax.set_title(f'r_top vs Subspace Divergence from DoRA  [r={R}]\n'
             'Each point = one layer', fontsize=11)
ax.legend(fontsize=8, ncol=2); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, f'fig4_rtop_vs_angle_r{R}.png'),
            dpi=150, bbox_inches='tight')
plt.show()

### 7.6 · All-threshold epoch dynamics (line plot)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.5))
for ti, (et, c) in enumerate(zip(THRESHOLDS, COLORS)):
    ax.plot(EPOCHS, et_epoch_mat[ti], 'o-', color=c, lw=2, ms=7,
            label=f'ET={et:.2f}')

ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Mean Principal Angle to DoRA final (°)', fontsize=12)
ax.set_title(f'Training Dynamics: ΔW Subspace Divergence from DoRA\n'
             f'All energy thresholds  [r={R}, k={K}]', fontsize=11)
ax.legend(fontsize=9, loc='best')
ax.grid(alpha=0.3); ax.set_xticks(EPOCHS)

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, f'fig5_et_epoch_dynamics_r{R}.png'),
            dpi=150, bbox_inches='tight')
plt.show()

### 7.7 · Effective rank of SALTEDORA ΔW vs threshold

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
mean_erank = np.nanmean(erank_mat, axis=1)
ax.plot(THRESHOLDS, mean_erank, 'D-', color='darkorange', lw=2, ms=8)
ax.fill_between(THRESHOLDS,
                np.nanmin(erank_mat, axis=1),
                np.nanmax(erank_mat, axis=1),
                alpha=0.15, color='darkorange', label='layer range')
ax.set_xlabel('Energy Threshold (ET)', fontsize=12)
ax.set_ylabel('Effective Rank of ΔW  (99% energy)', fontsize=12)
ax.set_title(f'Effective Rank of SALTEDORA ΔW vs Energy Threshold  [r={R}]', fontsize=11)
ax.legend(fontsize=9); ax.grid(alpha=0.3); ax.set_xticks(THRESHOLDS)

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, f'fig6_erank_vs_et_r{R}.png'),
            dpi=150, bbox_inches='tight')
plt.show()

---
## 8 · Numerical summary

In [ ]:
import pandas as pd

rows = []
for ti, et in enumerate(THRESHOLDS):
    rows.append({
        'Energy Threshold': f'{et:.2f}',
        'Mean Angle (°)':   round(float(np.nanmean(angle_mat[ti])), 2),
        'Min Angle (°)':    round(float(np.nanmin(angle_mat[ti])),  2),
        'Max Angle (°)':    round(float(np.nanmax(angle_mat[ti])),  2),
        'Cos² Alignment':   round(float(np.nanmean(align_mat[ti])), 4),
        'Mean r_top':       round(float(np.nanmean(rtop_mat[ti])),  1),
        'Mean Eff. Rank':   round(float(np.nanmean(erank_mat[ti])), 1),
    })

df_final = pd.DataFrame(rows)
print(f'\n=== Final Weights: SALTEDORA-V4 vs DoRA  [r={R}, k={K}] ===')
print(df_final.to_string(index=False))

# Per-epoch table for ET_DYN
rows_ep = []
for ei, ep in enumerate(EPOCHS):
    rows_ep.append({
        'Epoch':             ep,
        'Angle to DoRA (°)': round(float(np.nanmean(to_dora_mat[ei])), 2),
        'Consec. Angle (°)': round(float(np.nanmean(consec_mat[ei-1])), 2) if ei > 0 else '-',
    })

df_epoch = pd.DataFrame(rows_ep)
print(f'\n=== Epoch Dynamics  [r={R}, ET={ET_DYN}] ===')
print(df_epoch.to_string(index=False))

# All-threshold epoch table
et_ep_df = pd.DataFrame(
    np.round(et_epoch_mat, 2),
    index=[f'ET={t:.2f}' for t in THRESHOLDS],
    columns=[f'Ep{e}' for e in EPOCHS]
)
print(f'\n=== Angle to DoRA Final by Epoch & Threshold  [r={R}] ===')
print(et_ep_df.to_string())

---
## 9 · Optional: compare across ranks (fixed ET)

Run this cell to see how the principal angle changes as you increase the intrinsic rank `r`.

In [ ]:
RANKS_TO_TEST = [4, 8, 16, 32, 80, 96, 128, 160]
ET_FIXED      = 0.70

rank_angles = {}
for r in RANKS_TO_TEST:
    path = salt_pt(r, ET_FIXED, 'final')
    if not os.path.isfile(path):
        print(f'  [SKIP] r={r} not found'); continue
    salt_dws, _ = load_salt_dWs(path)
    angs = []
    for layer in LAYERS:
        dw_d = dora_dWs[layer]
        dw_s = salt_dws.get(layer)
        if dw_s is not None:
            try: angs.append(principal_angles_deg(dw_d, dw_s, K))
            except: pass
    del salt_dws; gc.collect()
    rank_angles[r] = np.nanmean(angs) if angs else np.nan
    print(f'  r={r:>4d}  mean_angle = {rank_angles[r]:.2f}°')

if rank_angles:
    rs   = list(rank_angles.keys())
    angs = [rank_angles[r] for r in rs]

    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.plot(rs, angs, 'o-', color='purple', lw=2, ms=8)
    ax.set_xlabel('Intrinsic Rank (r)', fontsize=12)
    ax.set_ylabel('Mean Principal Angle to DoRA (°)', fontsize=12)
    ax.set_title(f'Effect of Rank on Subspace Alignment with DoRA  [ET={ET_FIXED}]',
                 fontsize=11)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, f'fig7_rank_vs_angle_et{ET_FIXED:.2f}.png'),
                dpi=150, bbox_inches='tight')
    plt.show()

---
## 10 · Save raw arrays

In case you want to reload results later without rerunning everything.

In [ ]:
np.save(os.path.join(OUT_DIR, 'angles_matrix.npy'),   angle_mat)
np.save(os.path.join(OUT_DIR, 'align_matrix.npy'),    align_mat)
np.save(os.path.join(OUT_DIR, 'rtop_matrix.npy'),     rtop_mat)
np.save(os.path.join(OUT_DIR, 'erank_matrix.npy'),    erank_mat)
np.save(os.path.join(OUT_DIR, 'et_epoch_matrix.npy'), et_epoch_mat)
np.save(os.path.join(OUT_DIR, 'to_dora_matrix.npy'),  to_dora_mat)
np.save(os.path.join(OUT_DIR, 'consec_matrix.npy'),   consec_mat)

print('All arrays saved to:', OUT_DIR)